# 🚀 Huihui-Qwen3.8-27B-abliterated 云端一键极速部署

> **核心方案**：基于 `llama-server` 原生 CUDA 编译版 + `tool_sanitizer` 代理，采用 **Q4_K_XL / IQ3_XXS GGUF** 权重。
> **性能亮点**：显存占用仅 ~17GB，支持 **32K 超大安全上下文**，彻底解决 OOM 问题，速度提升数倍！
> **访问方式**：内置 Cloudflare Tunnel 自动内网穿透，一键生成专属公网 WebUI 与 OpenAI 兼容 API 地址（无需复杂端口映射）。

In [ ]:
#@title ⚡ 最终稳定版：强制使用 Q4_K_XL 模型 + 32K 上下文 (A100/L4 完美秒开)

import os, subprocess, time, re

# 1. 写入 tool_sanitizer.py
sanitizer_code = '''import json, os, httpx

UPSTREAM = os.environ.get("SANITIZER_UPSTREAM", "http://127.0.0.1:8000")
STRIP = {
    "maxLength", "minLength", "maxItems", "minItems", "maximum", "minimum",
    "exclusiveMaximum", "exclusiveMinimum", "pattern", "maxProperties",
    "minProperties", "multipleOf", "uniqueItems",
}
HOP = {"host", "content-length", "connection", "transfer-encoding", "keep-alive"}
client = httpx.AsyncClient(base_url=UPSTREAM, timeout=httpx.Timeout(3600, connect=30))

def strip(node, in_properties=False):
    if isinstance(node, dict):
        if in_properties:
            return {k: strip(v) for k, v in node.items()}
        out = {}
        for k, v in node.items():
            if k in STRIP and not isinstance(v, dict):
                continue
            out[k] = strip(v, in_properties=(k == "properties" and isinstance(v, dict)))
        return out
    if isinstance(node, list):
        return [strip(x) for x in node]
    return node

async def app(scope, receive, send):
    if scope["type"] != "http":
        return
    if scope["path"] in ("/", ""):
        await send({
            "type": "http.response.start",
            "status": 307,
            "headers": [(b"location", b"/qwen/")],
        })
        await send({"type": "http.response.body", "body": b""})
        return
    body = b""
    while True:
        msg = await receive()
        body += msg.get("body", b"")
        if not msg.get("more_body"):
            break
    if body:
        try:
            data = json.loads(body)
            if isinstance(data, dict) and data.get("tools"):
                data["tools"] = strip(data["tools"])
                body = json.dumps(data, ensure_ascii=False).encode()
        except ValueError:
            pass
    headers = [(k.decode(), v.decode()) for k, v in scope["headers"] if k.decode().lower() not in HOP]
    headers.append(("host", UPSTREAM.split("://", 1)[1]))
    path = scope["path"] + (("?" + scope["query_string"].decode()) if scope["query_string"] else "")
    req = client.build_request(scope["method"], path, headers=headers, content=body)
    resp = await client.send(req, stream=True)
    try:
        await send({
            "type": "http.response.start",
            "status": resp.status_code,
            "headers": [(k.encode(), v.encode()) for k, v in resp.headers.items() if k.lower() not in HOP],
        })
        async for chunk in resp.aiter_raw():
            await send({"type": "http.response.body", "body": chunk, "more_body": True})
        await send({"type": "http.response.body", "body": b""})
    finally:
        await resp.aclose()
'''
with open("/content/tool_sanitizer.py", "w") as f:
    f.write(sanitizer_code)

# 2. 写入引导脚本 (强制采用高性价比的 Q4_K_XL，彻底杜绝显存溢出)
bootstrap_script = r'''#!/bin/bash
set -uo pipefail

HF_REPO=huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF
MMPROJ=mmproj-model-bf16.gguf
LOG=/content/logs
BIN=/content/llama-bin
MODELS=/content/models
mkdir -p "$LOG" "$MODELS"
export LD_LIBRARY_PATH=/usr/lib64-nvidia:${LD_LIBRARY_PATH:-}

if ! nvidia-smi >/dev/null 2>&1; then
    echo "❌ 未检测到 GPU！"
    exit 1
fi

SM=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
VRAM=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -1)
GPU_NAME=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -1)

# 【强制优化】放弃容易 OOM 的 Q8，选用完美平衡显存与精度的 Q4_K_XL，显存占用仅 ~17GB，配合 32K 上下文绝对安全
if (( VRAM >= 22000 )); then
    MODEL_FILE=Huihui-Qwen3.8-27B-abliterated-UD-Q4_K_XL.gguf
    CTX=32768
else
    MODEL_FILE=Huihui-Qwen3.8-27B-abliterated-UD-IQ3_XXS.gguf
    CTX=16384
fi

echo "✅ 检测到显卡: $GPU_NAME (显存: ${VRAM}MiB, 架构: sm_${SM})"
echo "📦 自动匹配最稳模型: $MODEL_FILE (安全上下文: $CTX)"

echo "⏳ [并行任务 1] 正在从 Hugging Face 检查/下载 GGUF 模型..."
pip -q install -U huggingface_hub hf_xet uvicorn httpx >/dev/null 2>&1
( hf download "$HF_REPO" "$MODEL_FILE" "$MMPROJ" --local-dir "$MODELS" >"$LOG/dl.log" 2>&1 && touch "$LOG/dl.ok" ) &
DL_PID=$!

if [[ ! -x "$BIN/llama-server" ]]; then
    if curl -fsSL "https://github.com/AdamsYi/colab-qwen/releases/download/llama-bin/llama-bin-sm$SM.tar.gz" -o /content/llama-bin.tar.gz 2>/dev/null; then
        mkdir -p "$BIN" && tar -xzf /content/llama-bin.tar.gz -C "$BIN"
        echo "✅ 已直接下载预编译好的 llama-server (sm_$SM)！"
    fi
fi

if ! LD_LIBRARY_PATH=$BIN:$LD_LIBRARY_PATH "$BIN/llama-server" --version >/dev/null 2>&1; then
    echo "⏳ [并行任务 2] 正在源码编译 llama-server (约 5 分钟)..."
    rm -rf "$BIN" /content/llama.cpp
    export PATH=/usr/local/cuda/bin:$PATH CUDACXX=/usr/local/cuda/bin/nvcc
    git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp >"$LOG/build.log" 2>&1 \
    && cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON \
        -DCMAKE_CUDA_ARCHITECTURES="$SM" -DLLAMA_CURL=OFF >>"$LOG/build.log" 2>&1 \
    && cmake --build /content/llama.cpp/build --config Release -j "$(nproc)" --target llama-server >>"$LOG/build.log" 2>&1 \
    || { echo "❌ 编译失败："; tail -n 20 "$LOG/build.log"; exit 1; }
    mkdir -p "$BIN" && cp -a /content/llama.cpp/build/bin/. "$BIN/"
fi

echo "⏳ 等待模型下载完成..."
wait "$DL_PID"
[[ -f "$LOG/dl.ok" ]] || { echo "❌ 模型下载失败："; cat "$LOG/dl.log"; exit 1; }
echo "✅ 模型下载完成！"

echo "🚀 正在启动 llama-server 并加载模型进显存..."
pkill -x llama-server 2>/dev/null || true
pkill -f 'tool_sanitizer:app' 2>/dev/null || true
sleep 1

cd /content
LD_LIBRARY_PATH=$BIN:$LD_LIBRARY_PATH setsid nohup "$BIN/llama-server" \
    -m "$MODELS/$MODEL_FILE" --mmproj "$MODELS/$MMPROJ" \
    --host 127.0.0.1 --port 8000 -c "$CTX" --parallel 1 -ngl 999 --flash-attn on \
    --cache-type-k q8_0 --cache-type-v q8_0 --jinja --no-context-shift \
    --alias Huihui-Qwen3.8-27B-abliterated --api-prefix /qwen \
    >"$LOG/server.log" 2>&1 </dev/null &

SANITIZER_UPSTREAM=http://127.0.0.1:8000 setsid nohup python3 -m uvicorn --app-dir /content \
    --host 127.0.0.1 --port 8001 --no-access-log tool_sanitizer:app \
    >"$LOG/sanitizer.log" 2>&1 </dev/null &

for _ in $(seq 1 120); do
    if curl -fs -m 2 http://127.0.0.1:8000/qwen/health >/dev/null; then
        echo "🎉 模型加载成功！服务已就绪！"
        exit 0
    fi
    pgrep -x llama-server >/dev/null || { echo "❌ llama-server 异常退出，日志："; tail -n 30 "$LOG/server.log"; exit 1; }
    sleep 3
done
echo "❌ 加载超时，日志："
tail -n 30 "$LOG/server.log"
exit 1
'''

with open("/content/run_llama.sh", "w") as f:
    f.write(bootstrap_script)
os.chmod("/content/run_llama.sh", 0o755)

process = subprocess.Popen(["/bin/bash", "/content/run_llama.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode == 0:
    print("\n🌐 正在生成公网访问链接...")
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True)
    subprocess.run("pkill -f cloudflared 2>/dev/null || true", shell=True)
    tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8001"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    public_url = None
    for _ in range(60):
        line = tunnel.stdout.readline()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
            
    if public_url:
        print("="*65)
        print("🎉 部署成功！显存宽裕，运行流畅如飞！")
        print(f"👉 手机/电脑统一访问入口 : http://85.230.98.207:8765/ (会话持久保存)")
        print(f"👉 独立直接访问链接   : {public_url}/qwen/")
        print(f"💻 客户端 API 地址    : {public_url}/qwen/v1")
        print("   • 模型名称         : Huihui-Qwen3.8-27B-abliterated")
        print("="*65)
        
        import urllib.request, json
        print("\n⏳ 正在自动同步至你的手机端统一面板...")
        reg_api = "http://85.230.98.207:8765/api/qwen/register?key=chanson888"
        try:
            req = urllib.request.Request(
                reg_api,
                data=json.dumps({"url": public_url}).encode("utf-8"),
                headers={"Content-Type": "application/json"}
            )
            with urllib.request.urlopen(req, timeout=10) as resp:
                print("✅ 已成功注册到手机端！现在直接在手机浏览器打开 http://85.230.98.207:8765/ 即可使用 Qwen 27B，会话永久保存！")
        except Exception as e:
            print(f"⚠️ 自动同步提醒: 未能直接连通 PC 端 ({e})，你可以复制上方独立链接，或在手机端面板点击手动粘贴绑定。")
        
        print("\n🟢 服务持续守护运行中 (请保持此 Colab 标签页处于运行状态)...\n")
        try:
            tunnel.wait()
        except KeyboardInterrupt:
            print("\n🛑 用户已停止服务")
